In [1]:
from scipy.stats import ranksums

# Imports

import helpers.helper_functions as hf
import mne
import os.path as op
from mne.channels import combine_channels
import pandas as pd
from mne.beamformer import make_lcmv, apply_lcmv_epochs
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import hilbert
import helpers.test_circ_plot as circ_plot
import gc
import helpers.stc_helper as stc_helper
import time
from pycircstat2.hypothesis import rayleigh_test
ss = hf.settings_dict()

In [2]:
base_tmin = 0.0
base_tmax = 0.3
stim_tmin = 0.7
stim_tmax = 3.7

In [3]:
for subject_index in ss['subject_idx_list']:

    # loop over each event type
    for event_id in ss['event_id_list']:

        event_name = str(event_id)
        duty_cycle = ss['event_name_list'][event_id - 1]
        subjects_dir = ss['fs_subjects_dir']
        subject = ss['subject_id_list'][subject_index]
        print("loading dataset for subject: ", subject)

        save_dir = Path(ss['hilbert_ref_dir']) / subject / event_name
        save_dir.mkdir(parents=True, exist_ok=True)

        # load hilbert stc data
        hilbert_stc_file = Path(ss['hilbert_ref_dir']) / subject / event_name / f"{subject}-event-{event_name}-hilbert-ret-ref-vol.stc"
        stc = mne.read_source_estimate(hilbert_stc_file)
        stim_stc = stc.copy().crop(tmin=stim_tmin, tmax=stim_tmax)
        base_stc = stc.copy().crop(tmin=base_tmin, tmax=base_tmax)
        # stc.data shape: (n_voxels, n_times) — phase differences in radians [-pi, pi]
        n_voxels, n_times = stc.data.shape

        z_stats = np.zeros(n_voxels)
        p_vals = np.zeros(n_voxels)

        # Rayleigh test to get z-scores

        for voxel_idx in range(n_voxels):
            result = ranksums(stim_stc.data[voxel_idx], base_stc.data[voxel_idx])
            z_stats[voxel_idx], p_vals[voxel_idx] = result

        z_stc       = stc.copy().crop(0.7, 0.7 + 1/ss['fs'])
        z_stc.data  = z_stats[:, np.newaxis]   # (n_voxels, 1)

        z_stc.save(save_dir / f"{subject}-event-{event_name}-z-vol.stc" , overwrite=True)

        print(f"z min  : {z_stats.min():.4f}")
        print(f"z max  : {z_stats.max():.4f}")
        print(f"z mean : {z_stats.mean():.4f}")
        print(f"Voxels > 0   : {(z_stats > 0).sum()} / {n_voxels}")



loading dataset for subject:  0005_3SJ
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
z min  : -26.3411
z max  : 31.4146
z mean : -0.6557
Voxels > 0   : 672 / 1440
loading dataset for subject:  0005_3SJ
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
z min  : -27.9849
z max  : 32.1728
z mean : 0.1295
Voxels > 0   : 724 / 1440
loading dataset for subject:  0005_3SJ
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
z min  : -19.0573
z max  : 24.4851
z mean : 0.4143
Voxels > 0   : 708 / 1440
loading dataset for subject:  0005_3SJ
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
z min  : -28.3890
z max  : 22.3626
z mean : -1.4946
Voxels > 0   : 506 / 1440
loading dataset for subject:  0005_3SJ
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
z min  : -22.9952
z max  : 13.9253
z mean : -0.9836
Voxels > 0   : 558 / 14